In [6]:
"""
CFD: Laminar (streamline) flow vs Unsteady (vortex-shedding) flow
====================================================================
Same cylinder, same channel, same solver -- only the Reynolds number
(effectively the fluid's speed/viscosity ratio) differs. Visualized with
colored dye streaklines injected upstream, exactly how real wind tunnels
show flow behavior with colored smoke.

Method: incompressible Navier-Stokes ("stable fluids" scheme -- see
cylinder_flow.py for full derivation/comments), with a solid circular
obstacle. Colored dye is a passive scalar advected by the velocity field.

A note on "turbulence": genuine chaotic, small-scale turbulence is a
fundamentally 3D phenomenon (it requires vortex stretching, which does
not exist in 2D). Tested at both higher Reynolds number and doubled grid
resolution here, and in true 2D flow, energy organizes into large
coherent structures rather than cascading into fine-scale chaos -- this
is a real, known property of 2D fluid dynamics, not a bug. So the second
case here shows the physically-correct 2D result: unsteady, periodic
vortex shedding (the genuine onset of instability), rather than a faked
chaotic look.
"""

# ============================================================================
# IMPORTS
# ============================================================================
import os                              # used to build a file path that works
                                        # on any OS (Windows/Mac/Linux), and to
                                        # find "where is this script sitting"
import shutil                          # only used for shutil.which("ffmpeg"),
                                        # i.e. "is ffmpeg installed on this PC?"
import numpy as np                     # the actual math engine: every grid,
                                        # every velocity field, every bit of
                                        # arithmetic below is a numpy array
import matplotlib.pyplot as plt        # for building the figure/axes we draw
                                        # each animation frame into, and for
                                        # generating the rainbow color palette
import matplotlib.animation as animation  # turns a sequence of frames into an
                                        # .mp4 (or .gif) file
from matplotlib.patches import Circle  # draws the solid gray circle
                                        # representing the cylinder on top of
                                        # the flow field

# ============================================================================
# GRID / DOMAIN SETUP
# ============================================================================
# The simulation lives on a rectangular grid of numbers -- think of it as a
# piece of graph paper, where every little square holds a velocity value.
Nx, Ny = 300, 100      # interior grid resolution: 300 cells along the flow
                        # direction (x), 100 cells across the channel (y)
SX, SY = Nx + 2, Ny + 2 # actual array size = interior + 1 "ghost cell" on
                        # each side. Ghost cells are extra border cells used
                        # only to apply boundary conditions cleanly (so the
                        # interior math never needs special-case edge code)

D = 14.0                # cylinder diameter, in grid-cell units (not meters --
                         # this sim uses "grid units" throughout, not real
                         # physical units)
cx, cy = 45.0, Ny/2 + 2.0  # cylinder center: 45 cells in from the left edge,
                            # vertically just slightly off the exact centerline
                            # (that small offset is deliberate -- it "seeds" a
                            # tiny asymmetry so the wake has something to
                            # destabilize around, instead of staying perfectly
                            # symmetric forever)

U_in = 1.0   # inflow speed: fluid enters from the left at this fixed speed
             # every frame, forever. This is the "engine" driving everything.
dt = 0.5     # the simulation's time step -- how much simulated time passes
             # per physics update. (This scheme is unconditionally stable, so
             # a fairly large dt like this doesn't cause blow-ups, just some
             # loss of fine detail -- see cylinder_flow.py for more on this.)
GS_ITERS = 40  # number of Gauss-Seidel/Jacobi relaxation sweeps used inside
               # the iterative equation solvers below (more iterations = more
               # accurate solve, but slower; 40 is a practical middle ground)

# Build the coordinate grid, then mark every cell that falls inside the
# cylinder's circle as "solid" (mask=True). This mask is what turns an
# otherwise-empty rectangular channel into "channel with an obstacle in it".
xx, yy = np.meshgrid(np.arange(SX), np.arange(SY), indexing='ij')
mask = (xx - cx)**2 + (yy - cy)**2 <= (D/2)**2   # standard circle equation:
                                                   # distance-from-center <= radius


# ============================================================================
# BOUNDARY CONDITIONS
# ============================================================================
# These two functions run after every velocity/pressure update and "patch up"
# the four edges of the domain so they behave the way we want:
#   left   = fluid forced in at fixed speed  (inflow)
#   right  = fluid allowed to leave freely   (outflow)
#   top/bottom = fluid can't cross through, but isn't slowed down either
#                (frictionless "slip" walls)

def set_bnd_vel(u, v):
    """Boundary conditions for the velocity field (u = horizontal speed,
    v = vertical speed), applied after every velocity solve."""
    u[0, :] = U_in       # left edge: force horizontal speed to the fixed inflow speed
    v[0, :] = 0.0        # left edge: force vertical speed to zero (flow enters straight in)
    u[-1, :] = u[-2, :]  # right edge: copy the neighboring interior value
                          # (this is "zero gradient" -- fluid just exits
                          # however it's already moving, no artificial resistance)
    v[-1, :] = v[-2, :]  # same idea for vertical speed at the right edge
    v[:, 0] = 0.0         # bottom wall: vertical speed = 0 (fluid can't cross the wall)
    v[:, -1] = 0.0        # top wall: same -- can't cross through the ceiling
    u[:, 0] = u[:, 1]      # bottom wall: horizontal speed just copies the cell
                            # above it -- this is what makes it "slip" (frictionless):
                            # the fluid isn't dragged/slowed by touching the wall
    u[:, -1] = u[:, -2]    # same idea at the top wall


def set_bnd_pressure(p):
    """Boundary conditions for the pressure field, applied while solving
    the pressure equation (used to keep the fluid incompressible)."""
    p[0, :] = p[1, :]    # left (inflow): pressure has zero gradient here
                          # (because we already fixed the *velocity* there,
                          # we don't also constrain pressure independently)
    p[-1, :] = 0.0        # right (outflow): pressure fixed at a reference
                          # value of 0 -- this "anchors" the whole pressure
                          # field (pressure is only meaningful as differences,
                          # so one point has to be pinned down)
    p[:, 0] = p[:, 1]     # bottom wall: zero gradient (no flow through it, so
                          # no pressure build-up perpendicular to the wall)
    p[:, -1] = p[:, -2]   # top wall: same idea


# ============================================================================
# CORE NAVIER-STOKES SOLVER PIECES
# ============================================================================
# Together these four functions are one full "stable fluids" time step:
# diffuse (viscosity) -> project (incompressibility) -> advect (transport) ->
# project again. This exact pattern is explained in much more depth in the
# comments of cylinder_flow.py; here's the short version of each piece.

def diffuse_component(x, x0, diff_rate, dt, is_u, iters=GS_ITERS):
    """Applies viscosity: spreads/smooths out a velocity component (x) over
    time, based on how it looked before (x0) and how viscous the fluid is
    (diff_rate = nu). Solved implicitly via repeated relaxation sweeps
    (Gauss-Seidel style), which is what makes this scheme stable even with
    a fairly large dt."""
    a = dt * diff_rate  # combined "how much diffusion happens this step"
                         # factor (dx = 1 grid unit, so no extra /dx^2 needed)
    c_inv = 1.0 / (1 + 4*a)   # precompute this once outside the loop for speed
    for _ in range(iters):
        # each interior cell becomes a blend of its original value (x0) and
        # the average of its 4 neighbors (up/down/left/right) -- this is the
        # discrete version of the diffusion/heat equation
        x[1:-1, 1:-1] = (x0[1:-1, 1:-1] + a*(
            x[0:-2, 1:-1] + x[2:, 1:-1] + x[1:-1, 0:-2] + x[1:-1, 2:]
        )) * c_inv
        # re-apply boundary conditions after every single sweep, otherwise
        # the edges would drift away from the correct values as the
        # relaxation iterates
        if is_u:
            x[0, :] = U_in; x[-1, :] = x[-2, :]; x[:, 0] = x[:, 1]; x[:, -1] = x[:, -2]
        else:
            x[0, :] = 0.0; x[-1, :] = x[-2, :]; x[:, 0] = 0.0; x[:, -1] = 0.0
    return x


def advect(d, d0, u, v, dt, is_u):
    """Transports a velocity component (d) along the flow itself: for every
    grid cell, trace backward along the velocity field to find 'where did
    the fluid now sitting here come from a moment ago', then copy that
    earlier value forward. This is the 'semi-Lagrangian' advection scheme --
    it's what makes the whole method unconditionally stable."""
    # Xg, Yg = the (x, y) grid coordinate of every interior cell
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    # step backward along the velocity field by one time step, to find the
    # "source" location each cell's fluid came from
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    # clamp so we never sample outside the valid grid area
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    # since that source location generally falls *between* grid cells, find
    # the surrounding 4 integer grid cells (i0,j0)-(i1,j1) plus how far
    # between them we are (s0/s1, t0/t1), for bilinear interpolation
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    # bilinear blend of the 4 surrounding old values = the interpolated
    # "value that was there a moment ago"
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    # re-apply the same boundary conditions as before
    if is_u:
        d[0, :] = U_in; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    else:
        d[0, :] = 0.0; d[-1, :] = d[-2, :]; d[:, 0] = 0.0; d[:, -1] = 0.0
    return d


def advect_scalar(d, d0, u, v, dt):
    """Identical idea to advect() above, but for a plain scalar quantity
    (the colored dye) instead of a velocity component -- so its boundary
    handling is simpler: just 'copy the neighbor' (zero-gradient/open) on
    all four sides, since dye isn't a directional velocity that needs
    special inflow/wall treatment."""
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    d[0, :] = d[1, :]; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    return d


def project(u, v, p, div):
    """Enforces incompressibility (real fluids can't be squeezed into less
    space than they take up). Measures how much each cell's velocity is
    'diverging' (spreading outward = compressing/expanding), solves a
    pressure field that would exactly cancel that out, then subtracts the
    pressure's gradient from the velocity. This is the step that keeps the
    whole simulation physically sensible instead of just an arbitrary
    smoothed/advected mess."""
    # divergence of the velocity field: how much net outflow each cell has
    div[1:-1, 1:-1] = -0.5*(u[2:, 1:-1] - u[0:-2, 1:-1] + v[1:-1, 2:] - v[1:-1, 0:-2])
    p[:] = 0   # start the pressure guess at zero before solving
    set_bnd_pressure(div); set_bnd_pressure(p)
    # solve the pressure Poisson equation (∇²p = div) via repeated
    # relaxation sweeps -- same style of iterative solve as diffuse_component
    for _ in range(GS_ITERS):
        p[1:-1, 1:-1] = (div[1:-1, 1:-1] + p[0:-2, 1:-1] + p[2:, 1:-1] +
                          p[1:-1, 0:-2] + p[1:-1, 2:]) / 4.0
        set_bnd_pressure(p)
    # subtract the pressure gradient from velocity -- this is what actually
    # removes the divergence and makes the flow incompressible
    u[1:-1, 1:-1] -= 0.5*(p[2:, 1:-1] - p[0:-2, 1:-1])
    v[1:-1, 1:-1] -= 0.5*(p[1:-1, 2:] - p[1:-1, 0:-2])
    set_bnd_vel(u, v)
    return u, v


# ============================================================================
# DYE STREAKLINE SETUP (the colored lines you actually see in the video)
# ============================================================================
# 9 parallel streaklines are injected across the inlet, each a different
# color of the rainbow -- like 9 thin threads of colored smoke released into
# a wind tunnel, evenly spaced from just below the top wall to just above
# the bottom wall.
n_lines = 9
line_centers = np.linspace(8, Ny-6, n_lines)               # the y-position (row) of each line
line_colors = plt.cm.rainbow(np.linspace(0, 1, n_lines))[:, :3]  # an RGB color per line,
                                                                   # evenly spread across
                                                                   # the rainbow colormap


# ============================================================================
# MAIN SIMULATION + RENDERING FUNCTION
# ============================================================================
def run_case(Re, N_FRAMES, substeps, out_name, seed_perturbation, dye_decay=0.988):
    """Runs one full simulation (one Reynolds-number 'case') from a resting
    start all the way to a saved video file.

    Re               -- Reynolds number: sets how viscous the fluid behaves.
                         Low Re = thick/syrupy (laminar). High Re = thin/
                         energetic (unstable, sheds vortices).
    N_FRAMES         -- how many video frames to render.
    substeps         -- how many physics steps happen per rendered frame
                         (more substeps = smoother physics per frame, at
                         the cost of more compute).
    out_name         -- output filename (without extension).
    seed_perturbation-- whether to give the wake a small deliberate nudge
                         early on, to kick-start vortex shedding (only
                         needed for the unsteady/high-Re case).
    dye_decay        -- multiplier applied to the dye every step so old
                         dye slowly fades instead of piling up forever.
    """
    nu = U_in * D / Re     # convert the Reynolds number into an actual
                            # viscosity value the solver can use
                            # (Re = speed * size / viscosity, rearranged)
    print(f"\n=== {out_name}: Re={Re}, nu={nu:.4f} ===")

    # ---- initialize the flow field ----
    u = np.full((SX, SY), U_in)   # start with fluid already moving uniformly
                                    # at the inflow speed everywhere...
    v = np.zeros((SX, SY))         # ...with no vertical motion yet
    u[mask] = 0.0                  # ...except inside the cylinder, which is solid
    dyeR = np.zeros((SX, SY)); dyeG = np.zeros((SX, SY)); dyeB = np.zeros((SX, SY))
    # three separate scalar fields, one per color channel (Red/Green/Blue),
    # each advected exactly like a dye/smoke density -- together they form
    # a full-color image once combined

    frames = []       # will collect one rendered image per frame, to be
                       # assembled into the video at the end
    step_count = 0     # counts total physics steps taken (used for print logs)

    # ---- main time-stepping loop ----
    for frame in range(N_FRAMES):
        for sub in range(substeps):

            # --- Step A: viscosity (diffuse) ---
            u0 = u.copy(); v0 = v.copy()          # remember the "before" state
            diffuse_component(u, u0, nu, dt, True)
            diffuse_component(v, v0, nu, dt, False)

            # --- Step B: enforce incompressibility ---
            p = np.zeros((SX, SY)); div = np.zeros((SX, SY))
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0   # re-zero velocity inside the solid
                                             # cylinder (the "immersed boundary"
                                             # trick -- this is what makes the
                                             # obstacle actually block flow)

            # --- Step C: transport (advect) the velocity field along itself ---
            u0 = u.copy(); v0 = v.copy()
            advect(u, u0, u0, v0, dt, True)
            advect(v, v0, u0, v0, dt, False)

            # --- Step D: enforce incompressibility again (after advecting) ---
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- optional: nudge the wake to kick-start vortex shedding ---
            # (only used for the unsteady/high-Re case -- a real wind tunnel
            # always has tiny natural disturbances that do this job for free;
            # here we do it deliberately so shedding starts promptly instead
            # of waiting on floating-point rounding noise alone)
            if seed_perturbation and step_count < 8:
                v[int(cx)+3:int(cx)+8, Ny//2:Ny//2+2] += 0.3

            # --- inject fresh colored dye at the inlet, one thin band per line ---
            for yc, col in zip(line_centers, line_colors):
                j0, j1 = int(yc), int(yc)+1     # a 1-cell-thick horizontal band
                dyeR[1:3, j0:j1] += col[0]*0.42  # add this line's share of red
                dyeG[1:3, j0:j1] += col[1]*0.42  # ...green
                dyeB[1:3, j0:j1] += col[2]*0.42  # ...and blue

            # --- carry the dye along with the flow (passive advection) ---
            dR0 = dyeR.copy(); dG0 = dyeG.copy(); dB0 = dyeB.copy()
            advect_scalar(dyeR, dR0, u, v, dt)
            advect_scalar(dyeG, dG0, u, v, dt)
            advect_scalar(dyeB, dB0, u, v, dt)

            # --- fade the dye slightly each step, so old dye eventually ---
            # --- disappears instead of building up into a solid wash of color ---
            dyeR *= dye_decay; dyeG *= dye_decay; dyeB *= dye_decay
            dyeR[mask] = 0.0; dyeG[mask] = 0.0; dyeB[mask] = 0.0  # no dye inside the solid cylinder

            step_count += 1

        # ---- after all substeps for this frame, save a snapshot image ----
        rgb = np.stack([dyeR, dyeG, dyeB], axis=-1)[1:-1, 1:-1]  # combine the
                                                                    # 3 color
                                                                    # channels into
                                                                    # one RGB image,
                                                                    # dropping the
                                                                    # ghost-cell border
        frames.append(np.clip(rgb, 0, 1))   # clip so colors stay in the valid
                                              # 0-1 range for image display

        if frame % 40 == 0:
            print(f"  frame {frame}/{N_FRAMES}  t={step_count*dt:.0f}")

    print("  simulation complete, rendering...")

    # ========================================================================
    # RENDERING: turn the list of saved frames into an actual video file
    # ========================================================================
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")  # dark background
    ax.set_facecolor("#05070a")
    ax.axis("off")                        # hide axis ticks/labels -- just the flow image
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)  # use the entire figure area, no margins

    # draw the very first frame to set up the image object...
    im = ax.imshow(np.transpose(frames[0], (1, 0, 2)), origin="lower", interpolation="bilinear")
    # ...and draw the cylinder as a solid gray circle on top of it
    cyl_patch = Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a", edgecolor="#888888", linewidth=1, zorder=3)
    ax.add_patch(cyl_patch)

    def update(i):
        """Called once per frame by FuncAnimation: swap in the i-th saved
        frame's pixel data."""
        im.set_data(np.transpose(frames[i], (1, 0, 2)))
        return (im, cyl_patch)

    # build the animation object: calls update() for every frame, at roughly 30fps
    ani = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=1000/30, blit=True)

    # figure out where to save the output: next to this script if run as a
    # file, or the current folder if run inside a notebook (where __file__
    # doesn't exist)
    script_dir = os.getcwd() if "__file__" not in globals() else os.path.dirname(os.path.abspath(__file__))

    # check whether ffmpeg (needed to write .mp4) is installed on this machine
    has_ffmpeg = shutil.which("ffmpeg") is not None
    if has_ffmpeg:
        out_path = os.path.join(script_dir, out_name + ".mp4")
        writer = animation.FFMpegWriter(fps=30, bitrate=4000)
        ani.save(out_path, writer=writer, dpi=150)
    else:
        # fall back to an animated GIF if ffmpeg isn't available, so the
        # script still produces something viewable either way
        out_path = os.path.join(script_dir, out_name + ".gif")
        writer = animation.PillowWriter(fps=30)
        ani.save(out_path, writer=writer, dpi=100)

    plt.close(fig)   # free up memory/resources now that we're done with this figure
    print("  saved:", out_path)


# ============================================================================
# ENTRY POINT: run both regimes back-to-back when this file is executed directly
# ============================================================================
if __name__ == "__main__":
    # Case 1: laminar / streamline flow -- low Reynolds number, thick/viscous
    # fluid, no vortex shedding, streaklines stay smooth and never cross
    run_case(Re=30, N_FRAMES=200, substeps=3, out_name="flow_laminar_streamline",
              seed_perturbation=False, dye_decay=0.988)

    # Case 2: unsteady vortex-shedding flow -- high Reynolds number, thin/
    # energetic fluid, wake becomes unstable and sheds alternating vortices,
    # streaklines fold and mix downstream of the cylinder
    run_case(Re=10000, N_FRAMES=300, substeps=3, out_name="flow_unsteady_shedding",
              seed_perturbation=True, dye_decay=0.994)


=== flow_laminar_streamline: Re=30, nu=0.4667 ===
  frame 0/200  t=2
  frame 40/200  t=62
  frame 80/200  t=122
  frame 120/200  t=182
  frame 160/200  t=242
  simulation complete, rendering...
  saved: C:\Users\LENOVO\flow_laminar_streamline.gif

=== flow_unsteady_shedding: Re=10000, nu=0.0014 ===
  frame 0/300  t=2
  frame 40/300  t=62
  frame 80/300  t=122
  frame 120/300  t=182
  frame 160/300  t=242
  frame 200/300  t=302
  frame 240/300  t=362
  frame 280/300  t=422
  simulation complete, rendering...
  saved: C:\Users\LENOVO\flow_unsteady_shedding.gif


In [8]:
"""
CFD: Cylinder flow across a RANGE of Reynolds numbers (50 -> 10000)
====================================================================
Same base solver as before (incompressible "stable fluids" Navier-Stokes,
colored dye streaklines injected upstream). Now it:

  1. Loops over a LIST of Reynolds numbers instead of just two hard-coded
     cases, going from Re=50 (laminar / streamline) up to Re=10000
     (unsteady vortex shedding).
  2. Saves a PNG SNAPSHOT of the flow every `snapshot_every` frames for
     each case, in addition to the final .mp4/.gif video -- so you get a
     folder of still images showing how the flow develops over time, not
     just a video file.

Everything else (grid setup, boundary conditions, diffuse/advect/project)
is unchanged from the original script.
"""

# ============================================================================
# IMPORTS
# ============================================================================
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle

# ============================================================================
# GRID / DOMAIN SETUP
# ============================================================================
Nx, Ny = 300, 100
SX, SY = Nx + 2, Ny + 2

D = 14.0
cx, cy = 45.0, Ny/2 + 2.0

U_in = 1.0
dt = 0.5
GS_ITERS = 40

xx, yy = np.meshgrid(np.arange(SX), np.arange(SY), indexing='ij')
mask = (xx - cx)**2 + (yy - cy)**2 <= (D/2)**2


# ============================================================================
# BOUNDARY CONDITIONS
# ============================================================================
def set_bnd_vel(u, v):
    u[0, :] = U_in
    v[0, :] = 0.0
    u[-1, :] = u[-2, :]
    v[-1, :] = v[-2, :]
    v[:, 0] = 0.0
    v[:, -1] = 0.0
    u[:, 0] = u[:, 1]
    u[:, -1] = u[:, -2]


def set_bnd_pressure(p):
    p[0, :] = p[1, :]
    p[-1, :] = 0.0
    p[:, 0] = p[:, 1]
    p[:, -1] = p[:, -2]


# ============================================================================
# CORE NAVIER-STOKES SOLVER PIECES
# ============================================================================
def diffuse_component(x, x0, diff_rate, dt, is_u, iters=GS_ITERS):
    a = dt * diff_rate
    c_inv = 1.0 / (1 + 4*a)
    for _ in range(iters):
        x[1:-1, 1:-1] = (x0[1:-1, 1:-1] + a*(
            x[0:-2, 1:-1] + x[2:, 1:-1] + x[1:-1, 0:-2] + x[1:-1, 2:]
        )) * c_inv
        if is_u:
            x[0, :] = U_in; x[-1, :] = x[-2, :]; x[:, 0] = x[:, 1]; x[:, -1] = x[:, -2]
        else:
            x[0, :] = 0.0; x[-1, :] = x[-2, :]; x[:, 0] = 0.0; x[:, -1] = 0.0
    return x


def advect(d, d0, u, v, dt, is_u):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    if is_u:
        d[0, :] = U_in; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    else:
        d[0, :] = 0.0; d[-1, :] = d[-2, :]; d[:, 0] = 0.0; d[:, -1] = 0.0
    return d


def advect_scalar(d, d0, u, v, dt):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    d[0, :] = d[1, :]; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    return d


def project(u, v, p, div):
    div[1:-1, 1:-1] = -0.5*(u[2:, 1:-1] - u[0:-2, 1:-1] + v[1:-1, 2:] - v[1:-1, 0:-2])
    p[:] = 0
    set_bnd_pressure(div); set_bnd_pressure(p)
    for _ in range(GS_ITERS):
        p[1:-1, 1:-1] = (div[1:-1, 1:-1] + p[0:-2, 1:-1] + p[2:, 1:-1] +
                          p[1:-1, 0:-2] + p[1:-1, 2:]) / 4.0
        set_bnd_pressure(p)
    u[1:-1, 1:-1] -= 0.5*(p[2:, 1:-1] - p[0:-2, 1:-1])
    v[1:-1, 1:-1] -= 0.5*(p[1:-1, 2:] - p[1:-1, 0:-2])
    set_bnd_vel(u, v)
    return u, v


# ============================================================================
# DYE STREAKLINE SETUP
# ============================================================================
n_lines = 9
line_centers = np.linspace(8, Ny-6, n_lines)
line_colors = plt.cm.rainbow(np.linspace(0, 1, n_lines))[:, :3]


# ============================================================================
# SNAPSHOT HELPER
# ============================================================================
def save_snapshot(rgb, t, Re, frame_idx, snap_dir):
    """Saves a single still PNG of the current dye field + cylinder, exactly
    styled like a video frame. Called periodically from inside run_case()."""
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    ax.imshow(np.transpose(rgb, (1, 0, 2)), origin="lower", interpolation="bilinear")
    ax.add_patch(Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a",
                         edgecolor="#888888", linewidth=1, zorder=3))
    ax.text(0.01, 0.95, f"Re={Re}   t={t:.0f}   frame={frame_idx}",
            transform=ax.transAxes, color="white", fontsize=9,
            va="top", ha="left", family="monospace")

    fname = os.path.join(snap_dir, f"snap_frame{frame_idx:04d}_t{t:.0f}.png")
    fig.savefig(fname, dpi=150)
    plt.close(fig)


# ============================================================================
# MAIN SIMULATION + RENDERING FUNCTION
# ============================================================================
def run_case(Re, N_FRAMES, substeps, out_name, seed_perturbation,
             dye_decay=0.988, snapshot_every=20):
    """Runs one full simulation for a given Reynolds number, saving:
      - a snapshot PNG every `snapshot_every` frames (into <out_name>_snapshots/)
      - a final .mp4 (or .gif fallback) animation of the whole run

    snapshot_every -- how many rendered frames between saved snapshots.
                       e.g. 20 means a still image is written roughly every
                       20*substeps*dt units of simulated time.
    """
    nu = U_in * D / Re
    print(f"\n=== {out_name}: Re={Re}, nu={nu:.5f} ===")

    script_dir = os.getcwd() if "__file__" not in globals() else os.path.dirname(os.path.abspath(__file__))
    snap_dir = os.path.join(script_dir, out_name + "_snapshots")
    os.makedirs(snap_dir, exist_ok=True)

    # ---- initialize the flow field ----
    u = np.full((SX, SY), U_in)
    v = np.zeros((SX, SY))
    u[mask] = 0.0
    dyeR = np.zeros((SX, SY)); dyeG = np.zeros((SX, SY)); dyeB = np.zeros((SX, SY))

    frames = []
    step_count = 0

    # ---- main time-stepping loop ----
    for frame in range(N_FRAMES):
        for sub in range(substeps):
            # --- viscosity ---
            u0 = u.copy(); v0 = v.copy()
            diffuse_component(u, u0, nu, dt, True)
            diffuse_component(v, v0, nu, dt, False)

            # --- incompressibility ---
            p = np.zeros((SX, SY)); div = np.zeros((SX, SY))
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- advect velocity ---
            u0 = u.copy(); v0 = v.copy()
            advect(u, u0, u0, v0, dt, True)
            advect(v, v0, u0, v0, dt, False)

            # --- incompressibility again ---
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- optional wake nudge to kick-start shedding ---
            if seed_perturbation and step_count < 8:
                v[int(cx)+3:int(cx)+8, Ny//2:Ny//2+2] += 0.3

            # --- inject fresh dye ---
            for yc, col in zip(line_centers, line_colors):
                j0, j1 = int(yc), int(yc)+1
                dyeR[1:3, j0:j1] += col[0]*0.42
                dyeG[1:3, j0:j1] += col[1]*0.42
                dyeB[1:3, j0:j1] += col[2]*0.42

            # --- advect dye ---
            dR0 = dyeR.copy(); dG0 = dyeG.copy(); dB0 = dyeB.copy()
            advect_scalar(dyeR, dR0, u, v, dt)
            advect_scalar(dyeG, dG0, u, v, dt)
            advect_scalar(dyeB, dB0, u, v, dt)

            dyeR *= dye_decay; dyeG *= dye_decay; dyeB *= dye_decay
            dyeR[mask] = 0.0; dyeG[mask] = 0.0; dyeB[mask] = 0.0

            step_count += 1

        # ---- snapshot for this frame ----
        rgb = np.stack([dyeR, dyeG, dyeB], axis=-1)[1:-1, 1:-1]
        rgb = np.clip(rgb, 0, 1)
        frames.append(rgb)

        if frame % snapshot_every == 0:
            save_snapshot(rgb, step_count*dt, Re, frame, snap_dir)

        if frame % 40 == 0:
            print(f"  frame {frame}/{N_FRAMES}  t={step_count*dt:.0f}")

    print(f"  simulation complete. snapshots saved to: {snap_dir}")
    print("  rendering video...")

    # ========================================================================
    # RENDERING: turn the frame list into a video file
    # ========================================================================
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    im = ax.imshow(np.transpose(frames[0], (1, 0, 2)), origin="lower", interpolation="bilinear")
    cyl_patch = Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a", edgecolor="#888888", linewidth=1, zorder=3)
    ax.add_patch(cyl_patch)

    def update(i):
        im.set_data(np.transpose(frames[i], (1, 0, 2)))
        return (im, cyl_patch)

    ani = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=1000/30, blit=True)

    has_ffmpeg = shutil.which("ffmpeg") is not None
    if has_ffmpeg:
        out_path = os.path.join(script_dir, out_name + ".mp4")
        writer = animation.FFMpegWriter(fps=30, bitrate=4000)
        ani.save(out_path, writer=writer, dpi=150)
    else:
        out_path = os.path.join(script_dir, out_name + ".gif")
        writer = animation.PillowWriter(fps=30)
        ani.save(out_path, writer=writer, dpi=100)

    plt.close(fig)
    print("  saved:", out_path)


# ============================================================================
# ENTRY POINT: sweep Reynolds number from 50 up to 10000
# ============================================================================
if __name__ == "__main__":
    # Each entry: (Re, N_FRAMES, seed_perturbation, dye_decay)
    #   - low Re: thick/viscous, laminar, no perturbation needed
    #   - high Re: thin/energetic, needs a small nudge to kick off shedding,
    #     and a slower dye decay so the wake structure stays visible longer
    RE_CASES = [
        (50,    200, False, 0.988),
        (200,   220, False, 0.988),
        (1000,  250, True,  0.990),
        (5000,  280, True,  0.992),
        (10000, 300, True,  0.994),
    ]

    for Re, n_frames, seed, decay in RE_CASES:
        run_case(
            Re=Re,
            N_FRAMES=n_frames,
            substeps=3,
            out_name=f"flow_Re{Re}",
            seed_perturbation=seed,
            dye_decay=decay,
            snapshot_every=20,   # -> a PNG every 20 frames per case
        )

    print("\nAll cases complete.")


=== flow_Re50: Re=50, nu=0.28000 ===
  frame 0/200  t=2
  frame 40/200  t=62
  frame 80/200  t=122
  frame 120/200  t=182
  frame 160/200  t=242
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re50_snapshots
  rendering video...
  saved: C:\Users\LENOVO\flow_Re50.gif

=== flow_Re200: Re=200, nu=0.07000 ===
  frame 0/220  t=2
  frame 40/220  t=62
  frame 80/220  t=122
  frame 120/220  t=182
  frame 160/220  t=242
  frame 200/220  t=302
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re200_snapshots
  rendering video...
  saved: C:\Users\LENOVO\flow_Re200.gif

=== flow_Re1000: Re=1000, nu=0.01400 ===
  frame 0/250  t=2
  frame 40/250  t=62
  frame 80/250  t=122
  frame 120/250  t=182
  frame 160/250  t=242
  frame 200/250  t=302
  frame 240/250  t=362
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re1000_snapshots
  rendering video...
  saved: C:\Users\LENOVO\flow_Re1000.gif

=== flow_Re5000: Re=5000, nu=0.00280 ===
  frame 0/280  t=2
  f

In [9]:
"""
CFD: Cylinder flow across a RANGE of Reynolds numbers (50 -> 20000)
====================================================================
Same base solver as before (incompressible "stable fluids" Navier-Stokes,
colored dye streaklines injected upstream). Now it:

  1. Loops over a LIST of Reynolds numbers instead of just two hard-coded
     cases, going from Re=50 (laminar / streamline) up to Re=20000
     (unsteady vortex shedding, higher energy / finer wake structure).
  2. Saves a PNG SNAPSHOT of the flow every `snapshot_every` frames for
     each case, in addition to the final .mp4/.gif video -- so you get a
     folder of still images showing how the flow develops over time, not
     just a video file.

Everything else (grid setup, boundary conditions, diffuse/advect/project)
is unchanged from the original script.
"""

# ============================================================================
# IMPORTS
# ============================================================================
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle

# ============================================================================
# GRID / DOMAIN SETUP
# ============================================================================
Nx, Ny = 300, 100
SX, SY = Nx + 2, Ny + 2

D = 14.0
cx, cy = 45.0, Ny/2 + 2.0

U_in = 1.0
dt = 0.5
GS_ITERS = 40

xx, yy = np.meshgrid(np.arange(SX), np.arange(SY), indexing='ij')
mask = (xx - cx)**2 + (yy - cy)**2 <= (D/2)**2


# ============================================================================
# BOUNDARY CONDITIONS
# ============================================================================
def set_bnd_vel(u, v):
    u[0, :] = U_in
    v[0, :] = 0.0
    u[-1, :] = u[-2, :]
    v[-1, :] = v[-2, :]
    v[:, 0] = 0.0
    v[:, -1] = 0.0
    u[:, 0] = u[:, 1]
    u[:, -1] = u[:, -2]


def set_bnd_pressure(p):
    p[0, :] = p[1, :]
    p[-1, :] = 0.0
    p[:, 0] = p[:, 1]
    p[:, -1] = p[:, -2]


# ============================================================================
# CORE NAVIER-STOKES SOLVER PIECES
# ============================================================================
def diffuse_component(x, x0, diff_rate, dt, is_u, iters=GS_ITERS):
    a = dt * diff_rate
    c_inv = 1.0 / (1 + 4*a)
    for _ in range(iters):
        x[1:-1, 1:-1] = (x0[1:-1, 1:-1] + a*(
            x[0:-2, 1:-1] + x[2:, 1:-1] + x[1:-1, 0:-2] + x[1:-1, 2:]
        )) * c_inv
        if is_u:
            x[0, :] = U_in; x[-1, :] = x[-2, :]; x[:, 0] = x[:, 1]; x[:, -1] = x[:, -2]
        else:
            x[0, :] = 0.0; x[-1, :] = x[-2, :]; x[:, 0] = 0.0; x[:, -1] = 0.0
    return x


def advect(d, d0, u, v, dt, is_u):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    if is_u:
        d[0, :] = U_in; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    else:
        d[0, :] = 0.0; d[-1, :] = d[-2, :]; d[:, 0] = 0.0; d[:, -1] = 0.0
    return d


def advect_scalar(d, d0, u, v, dt):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    d[0, :] = d[1, :]; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    return d


def project(u, v, p, div):
    div[1:-1, 1:-1] = -0.5*(u[2:, 1:-1] - u[0:-2, 1:-1] + v[1:-1, 2:] - v[1:-1, 0:-2])
    p[:] = 0
    set_bnd_pressure(div); set_bnd_pressure(p)
    for _ in range(GS_ITERS):
        p[1:-1, 1:-1] = (div[1:-1, 1:-1] + p[0:-2, 1:-1] + p[2:, 1:-1] +
                          p[1:-1, 0:-2] + p[1:-1, 2:]) / 4.0
        set_bnd_pressure(p)
    u[1:-1, 1:-1] -= 0.5*(p[2:, 1:-1] - p[0:-2, 1:-1])
    v[1:-1, 1:-1] -= 0.5*(p[1:-1, 2:] - p[1:-1, 0:-2])
    set_bnd_vel(u, v)
    return u, v


# ============================================================================
# DYE STREAKLINE SETUP
# ============================================================================
n_lines = 9
line_centers = np.linspace(8, Ny-6, n_lines)
line_colors = plt.cm.rainbow(np.linspace(0, 1, n_lines))[:, :3]


# ============================================================================
# SNAPSHOT HELPER
# ============================================================================
def save_snapshot(rgb, t, Re, frame_idx, snap_dir):
    """Saves a single still PNG of the current dye field + cylinder, exactly
    styled like a video frame. Called periodically from inside run_case()."""
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    ax.imshow(np.transpose(rgb, (1, 0, 2)), origin="lower", interpolation="bilinear")
    ax.add_patch(Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a",
                         edgecolor="#888888", linewidth=1, zorder=3))
    ax.text(0.01, 0.95, f"Re={Re}   t={t:.0f}   frame={frame_idx}",
            transform=ax.transAxes, color="white", fontsize=9,
            va="top", ha="left", family="monospace")

    fname = os.path.join(snap_dir, f"snap_frame{frame_idx:04d}_t{t:.0f}.png")
    fig.savefig(fname, dpi=150)
    plt.close(fig)


# ============================================================================
# MAIN SIMULATION + RENDERING FUNCTION
# ============================================================================
def run_case(Re, N_FRAMES, substeps, out_name, seed_perturbation,
             dye_decay=0.988, snapshot_every=20):
    """Runs one full simulation for a given Reynolds number, saving:
      - a snapshot PNG every `snapshot_every` frames (into <out_name>_snapshots/)
      - a final .mp4 (or .gif fallback) animation of the whole run

    snapshot_every -- how many rendered frames between saved snapshots.
                       e.g. 20 means a still image is written roughly every
                       20*substeps*dt units of simulated time.
    """
    nu = U_in * D / Re
    print(f"\n=== {out_name}: Re={Re}, nu={nu:.5f} ===")

    script_dir = os.getcwd() if "__file__" not in globals() else os.path.dirname(os.path.abspath(__file__))
    snap_dir = os.path.join(script_dir, out_name + "_snapshots")
    os.makedirs(snap_dir, exist_ok=True)

    # ---- initialize the flow field ----
    u = np.full((SX, SY), U_in)
    v = np.zeros((SX, SY))
    u[mask] = 0.0
    dyeR = np.zeros((SX, SY)); dyeG = np.zeros((SX, SY)); dyeB = np.zeros((SX, SY))

    frames = []
    step_count = 0

    # ---- main time-stepping loop ----
    for frame in range(N_FRAMES):
        for sub in range(substeps):
            # --- viscosity ---
            u0 = u.copy(); v0 = v.copy()
            diffuse_component(u, u0, nu, dt, True)
            diffuse_component(v, v0, nu, dt, False)

            # --- incompressibility ---
            p = np.zeros((SX, SY)); div = np.zeros((SX, SY))
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- advect velocity ---
            u0 = u.copy(); v0 = v.copy()
            advect(u, u0, u0, v0, dt, True)
            advect(v, v0, u0, v0, dt, False)

            # --- incompressibility again ---
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- optional wake nudge to kick-start shedding ---
            if seed_perturbation and step_count < 8:
                v[int(cx)+3:int(cx)+8, Ny//2:Ny//2+2] += 0.3

            # --- inject fresh dye ---
            for yc, col in zip(line_centers, line_colors):
                j0, j1 = int(yc), int(yc)+1
                dyeR[1:3, j0:j1] += col[0]*0.42
                dyeG[1:3, j0:j1] += col[1]*0.42
                dyeB[1:3, j0:j1] += col[2]*0.42

            # --- advect dye ---
            dR0 = dyeR.copy(); dG0 = dyeG.copy(); dB0 = dyeB.copy()
            advect_scalar(dyeR, dR0, u, v, dt)
            advect_scalar(dyeG, dG0, u, v, dt)
            advect_scalar(dyeB, dB0, u, v, dt)

            dyeR *= dye_decay; dyeG *= dye_decay; dyeB *= dye_decay
            dyeR[mask] = 0.0; dyeG[mask] = 0.0; dyeB[mask] = 0.0

            step_count += 1

        # ---- snapshot for this frame ----
        rgb = np.stack([dyeR, dyeG, dyeB], axis=-1)[1:-1, 1:-1]
        rgb = np.clip(rgb, 0, 1)
        frames.append(rgb)

        if frame % snapshot_every == 0:
            save_snapshot(rgb, step_count*dt, Re, frame, snap_dir)

        if frame % 40 == 0:
            print(f"  frame {frame}/{N_FRAMES}  t={step_count*dt:.0f}")

    print(f"  simulation complete. snapshots saved to: {snap_dir}")

    # ---- final snapshot: last frame of this case, saved next to the video ----
    final_path = os.path.join(script_dir, out_name + "_final.png")
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    ax.imshow(np.transpose(frames[-1], (1, 0, 2)), origin="lower", interpolation="bilinear")
    ax.add_patch(Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a",
                         edgecolor="#888888", linewidth=1, zorder=3))
    ax.text(0.01, 0.95, f"Re={Re}   t={step_count*dt:.0f}   FINAL FRAME",
            transform=ax.transAxes, color="white", fontsize=9,
            va="top", ha="left", family="monospace")
    fig.savefig(final_path, dpi=150)
    plt.close(fig)
    print("  final snapshot saved:", final_path)

    print("  rendering video...")

    # ========================================================================
    # RENDERING: turn the frame list into a video file
    # ========================================================================
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    im = ax.imshow(np.transpose(frames[0], (1, 0, 2)), origin="lower", interpolation="bilinear")
    cyl_patch = Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a", edgecolor="#888888", linewidth=1, zorder=3)
    ax.add_patch(cyl_patch)

    def update(i):
        im.set_data(np.transpose(frames[i], (1, 0, 2)))
        return (im, cyl_patch)

    ani = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=1000/30, blit=True)

    has_ffmpeg = shutil.which("ffmpeg") is not None
    if has_ffmpeg:
        out_path = os.path.join(script_dir, out_name + ".mp4")
        writer = animation.FFMpegWriter(fps=30, bitrate=4000)
        ani.save(out_path, writer=writer, dpi=150)
    else:
        out_path = os.path.join(script_dir, out_name + ".gif")
        writer = animation.PillowWriter(fps=30)
        ani.save(out_path, writer=writer, dpi=100)

    plt.close(fig)
    print("  saved:", out_path)


# ============================================================================
# ENTRY POINT: sweep Reynolds number from 50 up to 20000
# ============================================================================
if __name__ == "__main__":
    # Each entry: (Re, N_FRAMES, seed_perturbation, dye_decay)
    #   - low Re: thick/viscous, laminar, no perturbation needed
    #   - high Re: thin/energetic, needs a small nudge to kick off shedding,
    #     and a slower dye decay so the wake structure stays visible longer
    RE_CASES = [
        (50,    200, False, 0.988),
        (200,   220, False, 0.988),
        (1000,  250, True,  0.990),
        (5000,  280, True,  0.992),
        (10000, 300, True,  0.994),
        (15000, 320, True,  0.995),
        (20000, 340, True,  0.996),
    ]

    for Re, n_frames, seed, decay in RE_CASES:
        run_case(
            Re=Re,
            N_FRAMES=n_frames,
            substeps=3,
            out_name=f"flow_Re{Re}",
            seed_perturbation=seed,
            dye_decay=decay,
            snapshot_every=20,   # -> a PNG every 20 frames per case
        )

    print("\nAll cases complete.")


=== flow_Re50: Re=50, nu=0.28000 ===
  frame 0/200  t=2
  frame 40/200  t=62
  frame 80/200  t=122
  frame 120/200  t=182
  frame 160/200  t=242
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re50_snapshots
  final snapshot saved: C:\Users\LENOVO\flow_Re50_final.png
  rendering video...
  saved: C:\Users\LENOVO\flow_Re50.gif

=== flow_Re200: Re=200, nu=0.07000 ===
  frame 0/220  t=2
  frame 40/220  t=62
  frame 80/220  t=122
  frame 120/220  t=182
  frame 160/220  t=242
  frame 200/220  t=302
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re200_snapshots
  final snapshot saved: C:\Users\LENOVO\flow_Re200_final.png
  rendering video...
  saved: C:\Users\LENOVO\flow_Re200.gif

=== flow_Re1000: Re=1000, nu=0.01400 ===
  frame 0/250  t=2
  frame 40/250  t=62
  frame 80/250  t=122
  frame 120/250  t=182
  frame 160/250  t=242
  frame 200/250  t=302
  frame 240/250  t=362
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re1000_snapshots
  fin

In [ ]:
"""
CFD: Cylinder flow across a RANGE of Reynolds numbers (50 -> 20000)
====================================================================
Same base solver as before (incompressible "stable fluids" Navier-Stokes,
colored dye streaklines injected upstream). Now it:

  1. Loops over a LIST of Reynolds numbers instead of just two hard-coded
     cases, going from Re=50 (laminar / streamline) up to Re=20000
     (unsteady vortex shedding, higher energy / finer wake structure).
  2. Saves a PNG SNAPSHOT of the flow every `snapshot_every` frames for
     each case, in addition to the final .mp4/.gif video -- so you get a
     folder of still images showing how the flow develops over time, not
     just a video file.

Everything else (grid setup, boundary conditions, diffuse/advect/project)
is unchanged from the original script.
"""

# ============================================================================
# IMPORTS
# ============================================================================
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle

# ============================================================================
# GRID / DOMAIN SETUP
# ============================================================================
Nx, Ny = 300, 100
SX, SY = Nx + 2, Ny + 2

D = 14.0
cx, cy = 45.0, Ny/2 + 2.0

U_in = 1.0
dt = 0.5
GS_ITERS = 40

xx, yy = np.meshgrid(np.arange(SX), np.arange(SY), indexing='ij')
mask = (xx - cx)**2 + (yy - cy)**2 <= (D/2)**2


# ============================================================================
# BOUNDARY CONDITIONS
# ============================================================================
def set_bnd_vel(u, v):
    u[0, :] = U_in
    v[0, :] = 0.0
    u[-1, :] = u[-2, :]
    v[-1, :] = v[-2, :]
    v[:, 0] = 0.0
    v[:, -1] = 0.0
    u[:, 0] = u[:, 1]
    u[:, -1] = u[:, -2]


def set_bnd_pressure(p):
    p[0, :] = p[1, :]
    p[-1, :] = 0.0
    p[:, 0] = p[:, 1]
    p[:, -1] = p[:, -2]


# ============================================================================
# CORE NAVIER-STOKES SOLVER PIECES
# ============================================================================
def diffuse_component(x, x0, diff_rate, dt, is_u, iters=GS_ITERS):
    a = dt * diff_rate
    c_inv = 1.0 / (1 + 4*a)
    for _ in range(iters):
        x[1:-1, 1:-1] = (x0[1:-1, 1:-1] + a*(
            x[0:-2, 1:-1] + x[2:, 1:-1] + x[1:-1, 0:-2] + x[1:-1, 2:]
        )) * c_inv
        if is_u:
            x[0, :] = U_in; x[-1, :] = x[-2, :]; x[:, 0] = x[:, 1]; x[:, -1] = x[:, -2]
        else:
            x[0, :] = 0.0; x[-1, :] = x[-2, :]; x[:, 0] = 0.0; x[:, -1] = 0.0
    return x


def advect(d, d0, u, v, dt, is_u):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    if is_u:
        d[0, :] = U_in; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    else:
        d[0, :] = 0.0; d[-1, :] = d[-2, :]; d[:, 0] = 0.0; d[:, -1] = 0.0
    return d


def advect_scalar(d, d0, u, v, dt):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    d[0, :] = d[1, :]; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    return d


def project(u, v, p, div):
    div[1:-1, 1:-1] = -0.5*(u[2:, 1:-1] - u[0:-2, 1:-1] + v[1:-1, 2:] - v[1:-1, 0:-2])
    p[:] = 0
    set_bnd_pressure(div); set_bnd_pressure(p)
    for _ in range(GS_ITERS):
        p[1:-1, 1:-1] = (div[1:-1, 1:-1] + p[0:-2, 1:-1] + p[2:, 1:-1] +
                          p[1:-1, 0:-2] + p[1:-1, 2:]) / 4.0
        set_bnd_pressure(p)
    u[1:-1, 1:-1] -= 0.5*(p[2:, 1:-1] - p[0:-2, 1:-1])
    v[1:-1, 1:-1] -= 0.5*(p[1:-1, 2:] - p[1:-1, 0:-2])
    set_bnd_vel(u, v)
    return u, v


# ============================================================================
# DYE STREAKLINE SETUP
# ============================================================================
n_lines = 9
line_centers = np.linspace(8, Ny-6, n_lines)
line_colors = plt.cm.rainbow(np.linspace(0, 1, n_lines))[:, :3]


# ============================================================================
# SNAPSHOT HELPER
# ============================================================================
def save_snapshot(rgb, t, Re, frame_idx, snap_dir):
    """Saves a single still PNG of the current dye field + cylinder, exactly
    styled like a video frame. Called periodically from inside run_case()."""
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    ax.imshow(np.transpose(rgb, (1, 0, 2)), origin="lower", interpolation="bilinear")
    ax.add_patch(Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a",
                         edgecolor="#888888", linewidth=1, zorder=3))

    fname = os.path.join(snap_dir, f"snap_frame{frame_idx:04d}_t{t:.0f}.png")
    fig.savefig(fname, dpi=150)
    plt.close(fig)


# ============================================================================
# MAIN SIMULATION + RENDERING FUNCTION
# ============================================================================
def run_case(Re, N_FRAMES, substeps, out_name, seed_perturbation,
             dye_decay=0.988, snapshot_every=20):
    """Runs one full simulation for a given Reynolds number, saving:
      - a snapshot PNG every `snapshot_every` frames (into <out_name>_snapshots/)
      - a final .mp4 (or .gif fallback) animation of the whole run

    snapshot_every -- how many rendered frames between saved snapshots.
                       e.g. 20 means a still image is written roughly every
                       20*substeps*dt units of simulated time.
    """
    nu = U_in * D / Re
    print(f"\n=== {out_name}: Re={Re}, nu={nu:.5f} ===")

    script_dir = os.getcwd() if "__file__" not in globals() else os.path.dirname(os.path.abspath(__file__))
    snap_dir = os.path.join(script_dir, out_name + "_snapshots")
    os.makedirs(snap_dir, exist_ok=True)

    # ---- initialize the flow field ----
    u = np.full((SX, SY), U_in)
    v = np.zeros((SX, SY))
    u[mask] = 0.0
    dyeR = np.zeros((SX, SY)); dyeG = np.zeros((SX, SY)); dyeB = np.zeros((SX, SY))

    frames = []
    step_count = 0

    # ---- main time-stepping loop ----
    for frame in range(N_FRAMES):
        for sub in range(substeps):
            # --- viscosity ---
            u0 = u.copy(); v0 = v.copy()
            diffuse_component(u, u0, nu, dt, True)
            diffuse_component(v, v0, nu, dt, False)

            # --- incompressibility ---
            p = np.zeros((SX, SY)); div = np.zeros((SX, SY))
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- advect velocity ---
            u0 = u.copy(); v0 = v.copy()
            advect(u, u0, u0, v0, dt, True)
            advect(v, v0, u0, v0, dt, False)

            # --- incompressibility again ---
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- optional wake nudge to kick-start shedding ---
            if seed_perturbation and step_count < 8:
                v[int(cx)+3:int(cx)+8, Ny//2:Ny//2+2] += 0.3

            # --- inject fresh dye ---
            for yc, col in zip(line_centers, line_colors):
                j0, j1 = int(yc), int(yc)+1
                dyeR[1:3, j0:j1] += col[0]*0.42
                dyeG[1:3, j0:j1] += col[1]*0.42
                dyeB[1:3, j0:j1] += col[2]*0.42

            # --- advect dye ---
            dR0 = dyeR.copy(); dG0 = dyeG.copy(); dB0 = dyeB.copy()
            advect_scalar(dyeR, dR0, u, v, dt)
            advect_scalar(dyeG, dG0, u, v, dt)
            advect_scalar(dyeB, dB0, u, v, dt)

            dyeR *= dye_decay; dyeG *= dye_decay; dyeB *= dye_decay
            dyeR[mask] = 0.0; dyeG[mask] = 0.0; dyeB[mask] = 0.0

            step_count += 1

        # ---- snapshot for this frame ----
        rgb = np.stack([dyeR, dyeG, dyeB], axis=-1)[1:-1, 1:-1]
        rgb = np.clip(rgb, 0, 1)
        frames.append(rgb)

        if frame % snapshot_every == 0:
            save_snapshot(rgb, step_count*dt, Re, frame, snap_dir)

        if frame % 40 == 0:
            print(f"  frame {frame}/{N_FRAMES}  t={step_count*dt:.0f}")

    print(f"  simulation complete. snapshots saved to: {snap_dir}")

    # ---- final snapshot: last frame of this case, saved next to the video ----
    final_path = os.path.join(script_dir, out_name + "_final.png")
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    ax.imshow(np.transpose(frames[-1], (1, 0, 2)), origin="lower", interpolation="bilinear")
    ax.add_patch(Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a",
                         edgecolor="#888888", linewidth=1, zorder=3))
    fig.savefig(final_path, dpi=150)
    plt.close(fig)
    print("  final snapshot saved:", final_path)

    print("  rendering video...")

    # ========================================================================
    # RENDERING: turn the frame list into a video file
    # ========================================================================
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    im = ax.imshow(np.transpose(frames[0], (1, 0, 2)), origin="lower", interpolation="bilinear")
    cyl_patch = Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a", edgecolor="#888888", linewidth=1, zorder=3)
    ax.add_patch(cyl_patch)

    def update(i):
        im.set_data(np.transpose(frames[i], (1, 0, 2)))
        return (im, cyl_patch)

    ani = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=1000/30, blit=True)

    has_ffmpeg = shutil.which("ffmpeg") is not None
    if has_ffmpeg:
        out_path = os.path.join(script_dir, out_name + ".mp4")
        writer = animation.FFMpegWriter(fps=30, bitrate=4000)
        ani.save(out_path, writer=writer, dpi=150)
    else:
        out_path = os.path.join(script_dir, out_name + ".gif")
        writer = animation.PillowWriter(fps=30)
        ani.save(out_path, writer=writer, dpi=100)

    plt.close(fig)
    print("  saved:", out_path)


# ============================================================================
# ENTRY POINT: sweep Reynolds number from 50 up to 20000
# ============================================================================
if __name__ == "__main__":
    # Each entry: (Re, N_FRAMES, seed_perturbation, dye_decay)
    #   - low Re: thick/viscous, laminar, no perturbation needed
    #   - high Re: thin/energetic, needs a small nudge to kick off shedding,
    #     and a slower dye decay so the wake structure stays visible longer
    RE_CASES = [
        (50,    200, False, 0.988),
        (200,   220, False, 0.988),
        (1000,  250, True,  0.990),
        (5000,  280, True,  0.992),
        (10000, 300, True,  0.994),
        (15000, 320, True,  0.995),
        (20000, 340, True,  0.996),
    ]

    for Re, n_frames, seed, decay in RE_CASES:
        run_case(
            Re=Re,
            N_FRAMES=n_frames,
            substeps=3,
            out_name=f"flow_Re{Re}",
            seed_perturbation=seed,
            dye_decay=decay,
            snapshot_every=20,   # -> a PNG every 20 frames per case
        )

    print("\nAll cases complete.")


=== flow_Re50: Re=50, nu=0.28000 ===
  frame 0/200  t=2
  frame 40/200  t=62
  frame 80/200  t=122
  frame 120/200  t=182
  frame 160/200  t=242
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re50_snapshots
  final snapshot saved: C:\Users\LENOVO\flow_Re50_final.png
  rendering video...
  saved: C:\Users\LENOVO\flow_Re50.gif

=== flow_Re200: Re=200, nu=0.07000 ===
  frame 0/220  t=2
  frame 40/220  t=62
  frame 80/220  t=122
  frame 120/220  t=182
  frame 160/220  t=242
  frame 200/220  t=302
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re200_snapshots
  final snapshot saved: C:\Users\LENOVO\flow_Re200_final.png
  rendering video...
  saved: C:\Users\LENOVO\flow_Re200.gif

=== flow_Re1000: Re=1000, nu=0.01400 ===
  frame 0/250  t=2
  frame 40/250  t=62
  frame 80/250  t=122
  frame 120/250  t=182
  frame 160/250  t=242
  frame 200/250  t=302
  frame 240/250  t=362
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re1000_snapshots
  fin

In [1]:
"""
CFD: Cylinder flow across a RANGE of Reynolds numbers (50 -> 20000)
====================================================================
Same base solver as before (incompressible "stable fluids" Navier-Stokes,
colored dye streaklines injected upstream). Now it:

  1. Loops over a LIST of Reynolds numbers instead of just two hard-coded
     cases, going from Re=50 (laminar / streamline) up to Re=20000
     (unsteady vortex shedding, higher energy / finer wake structure).
  2. Saves a PNG SNAPSHOT of the flow every `snapshot_every` frames for
     each case, in addition to the final .mp4/.gif video -- so you get a
     folder of still images showing how the flow develops over time, not
     just a video file.

Everything else (grid setup, boundary conditions, diffuse/advect/project)
is unchanged from the original script.
"""

# ============================================================================
# IMPORTS
# ============================================================================
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle

# ============================================================================
# GRID / DOMAIN SETUP
# ============================================================================
Nx, Ny = 300, 100
SX, SY = Nx + 2, Ny + 2

D = 14.0
cx, cy = 45.0, Ny/2 + 2.0

U_in = 1.0
dt = 0.5
GS_ITERS = 40

xx, yy = np.meshgrid(np.arange(SX), np.arange(SY), indexing='ij')
mask = (xx - cx)**2 + (yy - cy)**2 <= (D/2)**2


# ============================================================================
# BOUNDARY CONDITIONS
# ============================================================================
def set_bnd_vel(u, v):
    u[0, :] = U_in
    v[0, :] = 0.0
    u[-1, :] = u[-2, :]
    v[-1, :] = v[-2, :]
    v[:, 0] = 0.0
    v[:, -1] = 0.0
    u[:, 0] = u[:, 1]
    u[:, -1] = u[:, -2]


def set_bnd_pressure(p):
    p[0, :] = p[1, :]
    p[-1, :] = 0.0
    p[:, 0] = p[:, 1]
    p[:, -1] = p[:, -2]


# ============================================================================
# CORE NAVIER-STOKES SOLVER PIECES
# ============================================================================
def diffuse_component(x, x0, diff_rate, dt, is_u, iters=GS_ITERS):
    a = dt * diff_rate
    c_inv = 1.0 / (1 + 4*a)
    for _ in range(iters):
        x[1:-1, 1:-1] = (x0[1:-1, 1:-1] + a*(
            x[0:-2, 1:-1] + x[2:, 1:-1] + x[1:-1, 0:-2] + x[1:-1, 2:]
        )) * c_inv
        if is_u:
            x[0, :] = U_in; x[-1, :] = x[-2, :]; x[:, 0] = x[:, 1]; x[:, -1] = x[:, -2]
        else:
            x[0, :] = 0.0; x[-1, :] = x[-2, :]; x[:, 0] = 0.0; x[:, -1] = 0.0
    return x


def advect(d, d0, u, v, dt, is_u):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    if is_u:
        d[0, :] = U_in; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    else:
        d[0, :] = 0.0; d[-1, :] = d[-2, :]; d[:, 0] = 0.0; d[:, -1] = 0.0
    return d


def advect_scalar(d, d0, u, v, dt):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    d[0, :] = d[1, :]; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    return d


def project(u, v, p, div):
    div[1:-1, 1:-1] = -0.5*(u[2:, 1:-1] - u[0:-2, 1:-1] + v[1:-1, 2:] - v[1:-1, 0:-2])
    p[:] = 0
    set_bnd_pressure(div); set_bnd_pressure(p)
    for _ in range(GS_ITERS):
        p[1:-1, 1:-1] = (div[1:-1, 1:-1] + p[0:-2, 1:-1] + p[2:, 1:-1] +
                          p[1:-1, 0:-2] + p[1:-1, 2:]) / 4.0
        set_bnd_pressure(p)
    u[1:-1, 1:-1] -= 0.5*(p[2:, 1:-1] - p[0:-2, 1:-1])
    v[1:-1, 1:-1] -= 0.5*(p[1:-1, 2:] - p[1:-1, 0:-2])
    set_bnd_vel(u, v)
    return u, v


# ============================================================================
# DYE STREAKLINE SETUP
# ============================================================================
n_lines = 9
line_centers = np.linspace(8, Ny-6, n_lines)
line_colors = plt.cm.rainbow(np.linspace(0, 1, n_lines))[:, :3]


# ============================================================================
# SNAPSHOT HELPER
# ============================================================================
def save_snapshot(rgb, t, Re, frame_idx, snap_dir):
    """Saves a single still PNG of the current dye field + cylinder, exactly
    styled like a video frame. Called periodically from inside run_case()."""
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    ax.imshow(np.transpose(rgb, (1, 0, 2)), origin="lower", interpolation="bilinear")
    ax.add_patch(Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a",
                         edgecolor="#888888", linewidth=1, zorder=3))

    fname = os.path.join(snap_dir, f"snap_frame{frame_idx:04d}_t{t:.0f}.png")
    fig.savefig(fname, dpi=150)
    plt.close(fig)


# ============================================================================
# MAIN SIMULATION + RENDERING FUNCTION
# ============================================================================
def run_case(Re, N_FRAMES, substeps, out_name, seed_perturbation,
             dye_decay=0.988, snapshot_every=20):
    """Runs one full simulation for a given Reynolds number, saving:
      - a snapshot PNG every `snapshot_every` frames (into <out_name>_snapshots/)
      - a final .mp4 (or .gif fallback) animation of the whole run

    snapshot_every -- how many rendered frames between saved snapshots.
                       e.g. 20 means a still image is written roughly every
                       20*substeps*dt units of simulated time.
    """
    nu = U_in * D / Re
    print(f"\n=== {out_name}: Re={Re}, nu={nu:.5f} ===")

    script_dir = os.getcwd() if "__file__" not in globals() else os.path.dirname(os.path.abspath(__file__))
    snap_dir = os.path.join(script_dir, out_name + "_snapshots")
    os.makedirs(snap_dir, exist_ok=True)

    # ---- initialize the flow field ----
    u = np.full((SX, SY), U_in)
    v = np.zeros((SX, SY))
    u[mask] = 0.0
    dyeR = np.zeros((SX, SY)); dyeG = np.zeros((SX, SY)); dyeB = np.zeros((SX, SY))

    frames = []
    step_count = 0

    # ---- main time-stepping loop ----
    for frame in range(N_FRAMES):
        for sub in range(substeps):
            # --- viscosity ---
            u0 = u.copy(); v0 = v.copy()
            diffuse_component(u, u0, nu, dt, True)
            diffuse_component(v, v0, nu, dt, False)

            # --- incompressibility ---
            p = np.zeros((SX, SY)); div = np.zeros((SX, SY))
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- advect velocity ---
            u0 = u.copy(); v0 = v.copy()
            advect(u, u0, u0, v0, dt, True)
            advect(v, v0, u0, v0, dt, False)

            # --- incompressibility again ---
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- optional wake nudge to kick-start shedding ---
            if seed_perturbation and step_count < 8:
                v[int(cx)+3:int(cx)+8, Ny//2:Ny//2+2] += 0.3

            # --- inject fresh dye ---
            for yc, col in zip(line_centers, line_colors):
                j0, j1 = int(yc), int(yc)+1
                dyeR[1:3, j0:j1] += col[0]*0.42
                dyeG[1:3, j0:j1] += col[1]*0.42
                dyeB[1:3, j0:j1] += col[2]*0.42

            # --- advect dye ---
            dR0 = dyeR.copy(); dG0 = dyeG.copy(); dB0 = dyeB.copy()
            advect_scalar(dyeR, dR0, u, v, dt)
            advect_scalar(dyeG, dG0, u, v, dt)
            advect_scalar(dyeB, dB0, u, v, dt)

            dyeR *= dye_decay; dyeG *= dye_decay; dyeB *= dye_decay
            dyeR[mask] = 0.0; dyeG[mask] = 0.0; dyeB[mask] = 0.0

            step_count += 1

        # ---- snapshot for this frame ----
        rgb = np.stack([dyeR, dyeG, dyeB], axis=-1)[1:-1, 1:-1]
        rgb = np.clip(rgb, 0, 1)
        frames.append(rgb)

        if frame % snapshot_every == 0:
            save_snapshot(rgb, step_count*dt, Re, frame, snap_dir)

        if frame % 40 == 0:
            print(f"  frame {frame}/{N_FRAMES}  t={step_count*dt:.0f}")

    print(f"  simulation complete. snapshots saved to: {snap_dir}")

    # ---- final snapshot: last frame of this case, saved next to the video ----
    final_path = os.path.join(script_dir, out_name + "_final.png")
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    ax.imshow(np.transpose(frames[-1], (1, 0, 2)), origin="lower", interpolation="bilinear")
    ax.add_patch(Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a",
                         edgecolor="#888888", linewidth=1, zorder=3))
    fig.savefig(final_path, dpi=150)
    plt.close(fig)
    print("  final snapshot saved:", final_path)

    print("  rendering video...")

    # ========================================================================
    # RENDERING: turn the frame list into a video file
    # ========================================================================
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    im = ax.imshow(np.transpose(frames[0], (1, 0, 2)), origin="lower", interpolation="bilinear")
    cyl_patch = Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a", edgecolor="#888888", linewidth=1, zorder=3)
    ax.add_patch(cyl_patch)

    def update(i):
        im.set_data(np.transpose(frames[i], (1, 0, 2)))
        return (im, cyl_patch)

    ani = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=1000/30, blit=True)

    has_ffmpeg = shutil.which("ffmpeg") is not None
    if has_ffmpeg:
        out_path = os.path.join(script_dir, out_name + ".mp4")
        writer = animation.FFMpegWriter(fps=30, bitrate=4000)
        ani.save(out_path, writer=writer, dpi=150)
    else:
        out_path = os.path.join(script_dir, out_name + ".gif")
        writer = animation.PillowWriter(fps=30)
        ani.save(out_path, writer=writer, dpi=100)

    plt.close(fig)
    print("  saved:", out_path)


# ============================================================================
# ENTRY POINT: sweep Reynolds number from 50 up to 20000
# ============================================================================
if __name__ == "__main__":
    # Each entry: (Re, N_FRAMES, seed_perturbation, dye_decay)
    #   - low Re: thick/viscous, laminar, no perturbation needed
    #   - high Re: thin/energetic, needs a small nudge to kick off shedding,
    #     and a slower dye decay so the wake structure stays visible longer
    RE_CASES = [
        (50,    200, False, 0.988),
        (200,   220, False, 0.988),
        (1000,  250, True,  0.990),
        (5000,  280, True,  0.992),
        (10000, 300, True,  0.994),
        (15000, 320, True,  0.995),
        (20000, 340, True,  0.996),
    ]

    for Re, n_frames, seed, decay in RE_CASES:
        run_case(
            Re=Re,
            N_FRAMES=n_frames,
            substeps=3,
            out_name=f"flow_Re{Re}",
            seed_perturbation=seed,
            dye_decay=decay,
            snapshot_every=20,   # -> a PNG every 20 frames per case
        )

    print("\nAll cases complete.")


=== flow_Re50: Re=50, nu=0.28000 ===
  frame 0/200  t=2
  frame 40/200  t=62
  frame 80/200  t=122
  frame 120/200  t=182
  frame 160/200  t=242
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re50_snapshots
  final snapshot saved: C:\Users\LENOVO\flow_Re50_final.png
  rendering video...
  saved: C:\Users\LENOVO\flow_Re50.gif

=== flow_Re200: Re=200, nu=0.07000 ===
  frame 0/220  t=2
  frame 40/220  t=62
  frame 80/220  t=122
  frame 120/220  t=182
  frame 160/220  t=242
  frame 200/220  t=302
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re200_snapshots
  final snapshot saved: C:\Users\LENOVO\flow_Re200_final.png
  rendering video...
  saved: C:\Users\LENOVO\flow_Re200.gif

=== flow_Re1000: Re=1000, nu=0.01400 ===
  frame 0/250  t=2
  frame 40/250  t=62
  frame 80/250  t=122
  frame 120/250  t=182
  frame 160/250  t=242
  frame 200/250  t=302
  frame 240/250  t=362
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re1000_snapshots
  fin

In [1]:
"""
CFD: Cylinder flow across a RANGE of Reynolds numbers (50 -> 20000)
====================================================================
Same base solver as before (incompressible "stable fluids" Navier-Stokes,
colored dye streaklines injected upstream). Now it:

  1. Loops over a LIST of Reynolds numbers instead of just two hard-coded
     cases, going from Re=50 (laminar / streamline) up to Re=20000
     (unsteady vortex shedding, higher energy / finer wake structure).
  2. Saves a PNG SNAPSHOT of the flow every `snapshot_every` frames for
     each case, in addition to the final .mp4/.gif video -- so you get a
     folder of still images showing how the flow develops over time, not
     just a video file.

Everything else (grid setup, boundary conditions, diffuse/advect/project)
is unchanged from the original script.
"""

# ============================================================================
# IMPORTS
# ============================================================================
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle

# ============================================================================
# GRID / DOMAIN SETUP
# ============================================================================
Nx, Ny = 300, 100
SX, SY = Nx + 2, Ny + 2

D = 14.0
cx, cy = 45.0, Ny/2 + 2.0

U_in = 1.0
dt = 0.5
GS_ITERS = 40

xx, yy = np.meshgrid(np.arange(SX), np.arange(SY), indexing='ij')
mask = (xx - cx)**2 + (yy - cy)**2 <= (D/2)**2


# ============================================================================
# BOUNDARY CONDITIONS
# ============================================================================
def set_bnd_vel(u, v):
    u[0, :] = U_in
    v[0, :] = 0.0
    u[-1, :] = u[-2, :]
    v[-1, :] = v[-2, :]
    v[:, 0] = 0.0
    v[:, -1] = 0.0
    u[:, 0] = u[:, 1]
    u[:, -1] = u[:, -2]


def set_bnd_pressure(p):
    p[0, :] = p[1, :]
    p[-1, :] = 0.0
    p[:, 0] = p[:, 1]
    p[:, -1] = p[:, -2]


# ============================================================================
# CORE NAVIER-STOKES SOLVER PIECES
# ============================================================================
def diffuse_component(x, x0, diff_rate, dt, is_u, iters=GS_ITERS):
    a = dt * diff_rate
    c_inv = 1.0 / (1 + 4*a)
    for _ in range(iters):
        x[1:-1, 1:-1] = (x0[1:-1, 1:-1] + a*(
            x[0:-2, 1:-1] + x[2:, 1:-1] + x[1:-1, 0:-2] + x[1:-1, 2:]
        )) * c_inv
        if is_u:
            x[0, :] = U_in; x[-1, :] = x[-2, :]; x[:, 0] = x[:, 1]; x[:, -1] = x[:, -2]
        else:
            x[0, :] = 0.0; x[-1, :] = x[-2, :]; x[:, 0] = 0.0; x[:, -1] = 0.0
    return x


def advect(d, d0, u, v, dt, is_u):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    if is_u:
        d[0, :] = U_in; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    else:
        d[0, :] = 0.0; d[-1, :] = d[-2, :]; d[:, 0] = 0.0; d[:, -1] = 0.0
    return d


def advect_scalar(d, d0, u, v, dt):
    Xg, Yg = np.meshgrid(np.arange(1, Nx+1), np.arange(1, Ny+1), indexing='ij')
    x = Xg - dt*u[1:-1, 1:-1]; y = Yg - dt*v[1:-1, 1:-1]
    x = np.clip(x, 0.5, Nx+0.5); y = np.clip(y, 0.5, Ny+0.5)
    i0 = x.astype(int); i1 = i0+1; j0 = y.astype(int); j1 = j0+1
    s1 = x-i0; s0 = 1-s1; t1 = y-j0; t0 = 1-t1
    d[1:-1, 1:-1] = (s0*(t0*d0[i0, j0] + t1*d0[i0, j1]) +
                      s1*(t0*d0[i1, j0] + t1*d0[i1, j1]))
    d[0, :] = d[1, :]; d[-1, :] = d[-2, :]; d[:, 0] = d[:, 1]; d[:, -1] = d[:, -2]
    return d


def project(u, v, p, div):
    div[1:-1, 1:-1] = -0.5*(u[2:, 1:-1] - u[0:-2, 1:-1] + v[1:-1, 2:] - v[1:-1, 0:-2])
    p[:] = 0
    set_bnd_pressure(div); set_bnd_pressure(p)
    for _ in range(GS_ITERS):
        p[1:-1, 1:-1] = (div[1:-1, 1:-1] + p[0:-2, 1:-1] + p[2:, 1:-1] +
                          p[1:-1, 0:-2] + p[1:-1, 2:]) / 4.0
        set_bnd_pressure(p)
    u[1:-1, 1:-1] -= 0.5*(p[2:, 1:-1] - p[0:-2, 1:-1])
    v[1:-1, 1:-1] -= 0.5*(p[1:-1, 2:] - p[1:-1, 0:-2])
    set_bnd_vel(u, v)
    return u, v


# ============================================================================
# DYE STREAKLINE SETUP
# ============================================================================
n_lines = 9
line_centers = np.linspace(8, Ny-6, n_lines)
line_colors = plt.cm.rainbow(np.linspace(0, 1, n_lines))[:, :3]


# ============================================================================
# SNAPSHOT HELPER
# ============================================================================
def save_snapshot(rgb, t, Re, frame_idx, snap_dir):
    """Saves a single still PNG of the current dye field + cylinder, exactly
    styled like a video frame. Called periodically from inside run_case()."""
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    ax.imshow(np.transpose(rgb, (1, 0, 2)), origin="lower", interpolation="bilinear")
    ax.add_patch(Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a",
                         edgecolor="#888888", linewidth=1, zorder=3))
    ax.text(0.01, 0.95, f"Re={Re}",
            transform=ax.transAxes, color="white", fontsize=11,
            va="top", ha="left", family="monospace")

    fname = os.path.join(snap_dir, f"snap_frame{frame_idx:04d}_t{t:.0f}.png")
    fig.savefig(fname, dpi=150)
    plt.close(fig)


# ============================================================================
# MAIN SIMULATION + RENDERING FUNCTION
# ============================================================================
def run_case(Re, N_FRAMES, substeps, out_name, seed_perturbation,
             dye_decay=0.988, snapshot_every=20):
    """Runs one full simulation for a given Reynolds number, saving:
      - a snapshot PNG every `snapshot_every` frames (into <out_name>_snapshots/)
      - a final .mp4 (or .gif fallback) animation of the whole run

    snapshot_every -- how many rendered frames between saved snapshots.
                       e.g. 20 means a still image is written roughly every
                       20*substeps*dt units of simulated time.
    """
    nu = U_in * D / Re
    print(f"\n=== {out_name}: Re={Re}, nu={nu:.5f} ===")

    script_dir = os.getcwd() if "__file__" not in globals() else os.path.dirname(os.path.abspath(__file__))
    snap_dir = os.path.join(script_dir, out_name + "_snapshots")
    os.makedirs(snap_dir, exist_ok=True)

    # ---- initialize the flow field ----
    u = np.full((SX, SY), U_in)
    v = np.zeros((SX, SY))
    u[mask] = 0.0
    dyeR = np.zeros((SX, SY)); dyeG = np.zeros((SX, SY)); dyeB = np.zeros((SX, SY))

    frames = []
    step_count = 0

    # ---- main time-stepping loop ----
    for frame in range(N_FRAMES):
        for sub in range(substeps):
            # --- viscosity ---
            u0 = u.copy(); v0 = v.copy()
            diffuse_component(u, u0, nu, dt, True)
            diffuse_component(v, v0, nu, dt, False)

            # --- incompressibility ---
            p = np.zeros((SX, SY)); div = np.zeros((SX, SY))
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- advect velocity ---
            u0 = u.copy(); v0 = v.copy()
            advect(u, u0, u0, v0, dt, True)
            advect(v, v0, u0, v0, dt, False)

            # --- incompressibility again ---
            project(u, v, p, div)
            u[mask] = 0.0; v[mask] = 0.0

            # --- optional wake nudge to kick-start shedding ---
            if seed_perturbation and step_count < 8:
                v[int(cx)+3:int(cx)+8, Ny//2:Ny//2+2] += 0.3

            # --- inject fresh dye ---
            for yc, col in zip(line_centers, line_colors):
                j0, j1 = int(yc), int(yc)+1
                dyeR[1:3, j0:j1] += col[0]*0.42
                dyeG[1:3, j0:j1] += col[1]*0.42
                dyeB[1:3, j0:j1] += col[2]*0.42

            # --- advect dye ---
            dR0 = dyeR.copy(); dG0 = dyeG.copy(); dB0 = dyeB.copy()
            advect_scalar(dyeR, dR0, u, v, dt)
            advect_scalar(dyeG, dG0, u, v, dt)
            advect_scalar(dyeB, dB0, u, v, dt)

            dyeR *= dye_decay; dyeG *= dye_decay; dyeB *= dye_decay
            dyeR[mask] = 0.0; dyeG[mask] = 0.0; dyeB[mask] = 0.0

            step_count += 1

        # ---- snapshot for this frame ----
        rgb = np.stack([dyeR, dyeG, dyeB], axis=-1)[1:-1, 1:-1]
        rgb = np.clip(rgb, 0, 1)
        frames.append(rgb)

        if frame % snapshot_every == 0:
            save_snapshot(rgb, step_count*dt, Re, frame, snap_dir)

        if frame % 40 == 0:
            print(f"  frame {frame}/{N_FRAMES}  t={step_count*dt:.0f}")

    print(f"  simulation complete. snapshots saved to: {snap_dir}")

    # ---- final snapshot: last frame of this case, saved next to the video ----
    final_path = os.path.join(script_dir, out_name + "_final.png")
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    ax.imshow(np.transpose(frames[-1], (1, 0, 2)), origin="lower", interpolation="bilinear")
    ax.add_patch(Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a",
                         edgecolor="#888888", linewidth=1, zorder=3))
    ax.text(0.01, 0.95, f"Re={Re}",
            transform=ax.transAxes, color="white", fontsize=11,
            va="top", ha="left", family="monospace")
    fig.savefig(final_path, dpi=150)
    plt.close(fig)
    print("  final snapshot saved:", final_path)

    print("  rendering video...")

    # ========================================================================
    # RENDERING: turn the frame list into a video file
    # ========================================================================
    fig, ax = plt.subplots(figsize=(9, 3.2), facecolor="#05070a")
    ax.set_facecolor("#05070a")
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    im = ax.imshow(np.transpose(frames[0], (1, 0, 2)), origin="lower", interpolation="bilinear")
    cyl_patch = Circle((cx-1, cy-1), D/2, facecolor="#3a3a3a", edgecolor="#888888", linewidth=1, zorder=3)
    ax.add_patch(cyl_patch)

    def update(i):
        im.set_data(np.transpose(frames[i], (1, 0, 2)))
        return (im, cyl_patch)

    ani = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=1000/30, blit=True)

    has_ffmpeg = shutil.which("ffmpeg") is not None
    if has_ffmpeg:
        out_path = os.path.join(script_dir, out_name + ".mp4")
        writer = animation.FFMpegWriter(fps=30, bitrate=4000)
        ani.save(out_path, writer=writer, dpi=150)
    else:
        out_path = os.path.join(script_dir, out_name + ".gif")
        writer = animation.PillowWriter(fps=30)
        ani.save(out_path, writer=writer, dpi=100)

    plt.close(fig)
    print("  saved:", out_path)


# ============================================================================
# ENTRY POINT: sweep Reynolds number from 50 up to 20000
# ============================================================================
if __name__ == "__main__":
    # Each entry: (Re, N_FRAMES, seed_perturbation, dye_decay)
    #   - low Re: thick/viscous, laminar, no perturbation needed
    #   - high Re: thin/energetic, needs a small nudge to kick off shedding,
    #     and a slower dye decay so the wake structure stays visible longer
    RE_CASES = [
        (50,    200, False, 0.988),
        (200,   220, False, 0.988),
        (1000,  250, True,  0.990),
        (5000,  280, True,  0.992),
        (10000, 300, True,  0.994),
        (15000, 320, True,  0.995),
        (20000, 340, True,  0.996),
    ]

    for Re, n_frames, seed, decay in RE_CASES:
        run_case(
            Re=Re,
            N_FRAMES=n_frames,
            substeps=3,
            out_name=f"flow_Re{Re}",
            seed_perturbation=seed,
            dye_decay=decay,
            snapshot_every=20,   # -> a PNG every 20 frames per case
        )

    print("\nAll cases complete.")


=== flow_Re50: Re=50, nu=0.28000 ===
  frame 0/200  t=2
  frame 40/200  t=62
  frame 80/200  t=122
  frame 120/200  t=182
  frame 160/200  t=242
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re50_snapshots
  final snapshot saved: C:\Users\LENOVO\flow_Re50_final.png
  rendering video...
  saved: C:\Users\LENOVO\flow_Re50.gif

=== flow_Re200: Re=200, nu=0.07000 ===
  frame 0/220  t=2
  frame 40/220  t=62
  frame 80/220  t=122
  frame 120/220  t=182
  frame 160/220  t=242
  frame 200/220  t=302
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re200_snapshots
  final snapshot saved: C:\Users\LENOVO\flow_Re200_final.png
  rendering video...
  saved: C:\Users\LENOVO\flow_Re200.gif

=== flow_Re1000: Re=1000, nu=0.01400 ===
  frame 0/250  t=2
  frame 40/250  t=62
  frame 80/250  t=122
  frame 120/250  t=182
  frame 160/250  t=242
  frame 200/250  t=302
  frame 240/250  t=362
  simulation complete. snapshots saved to: C:\Users\LENOVO\flow_Re1000_snapshots
  fin